# Graded Assignment 2: MLOps DVC Setup and Versioning

This notebook guides you through setting up and running **DVC (Data Version Control)** in your Git repository `21f1005023_MLOPS_WEEKLY_ASSIGNMENT` on branch `week_2`.

## Assignment Tasks:
1. **Initialize DVC**: Initialize DVC inside your local repository and commit DVC config files to Git.
2. **Configure GCS Remote**: Create a Google Cloud Storage bucket and configure it as the default remote for DVC.
3. **Version Data & Models Across Iterations**: Create multiple versions of the dataset, train models for each, and track both data/models with DVC.
4. **Switch Between Versions**: Switch between versions using `git checkout` and `dvc checkout`, and verify the local files update correctly.
5. **Push All Relevant Files**: Push all config files, pointer files, and python scripts to the remote Git repository.

### Step 0: Install Dependencies

First, install DVC with Google Cloud Storage support (`dvc-gs` and `dvc[gs]`) and other necessary packages for training and storage client operations.

In [1]:
# Install DVC with GCS support and machine learning requirements
%pip install "dvc[gs]" dvc-gs pandas scikit-learn joblib google-cloud-storage

Note: you may need to restart the kernel to use updated packages.


### Task 1: Initialize DVC

Initialize DVC in the repository. Make sure you are in the root directory of your git repository when executing this command.

In [ ]:
# Initialize DVC
!dvc init --force

#### **Terminal Action Required**

After DVC is initialized, commit the configuration files to Git.
Run the following commands in your terminal:
```bash
git add .dvc/config .dvc/.gitignore .dvcignore
git commit -m "Initialize DVC"
```

### Task 2: Configure Google Cloud Storage as DVC Remote

First, run the Python code below to create/retrieve your GCS bucket. Configure your Project ID and preferred bucket name below.

*Note: Ensure you are authenticated with Google Cloud (e.g., via `gcloud auth application-default login` in your terminal).* 

In [2]:
import os
from google.cloud import storage

# --- Configure Google Cloud settings here ---
PROJECT_ID = "project-f9a302e4-48ab-4c0e-91b4"  # Replace with your actual Project ID
LOCATION = "us-central1"
BUCKET_NAME = "week2-dvc-bucket-21f1005023"     # Replace with your desired unique bucket name
BUCKET_URI = f"gs://{BUCKET_NAME}"

print(f"Project ID: {PROJECT_ID}")
print(f"Bucket URI: {BUCKET_URI}")

# Create GCS Bucket programmatically if it doesn't exist
storage_client = storage.Client(project=PROJECT_ID)
try:
    bucket = storage_client.lookup_bucket(BUCKET_NAME)
    if bucket is None:
        bucket = storage_client.create_bucket(BUCKET_NAME, location=LOCATION)
        print(f"Successfully created bucket: {BUCKET_NAME}")
    else:
        print(f"Bucket already exists: {BUCKET_NAME}")
except Exception as e:
    print(f"Warning: Could not check/create bucket programmatically: {e}")
    print("Ensure you have run `gcloud auth application-default login` and have correct permissions.")

Project ID: project-f9a302e4-48ab-4c0e-91b4
Bucket URI: gs://week2-dvc-bucket-21f1005023
Bucket already exists: week2-dvc-bucket-21f1005023


#### Add and configure default GCS remote for DVC

Now we register the GCS bucket as our DVC default remote.

In [4]:
# Configure GCS remote
print(f"dvc remote add -d -f gcs-remote gs://{BUCKET_NAME}/dvc")

dvc remote add -d -f gcs-remote gs://week2-dvc-bucket-21f1005023/dvc


#### **Terminal Action Required**

Commit `.dvc/config` to Git:
```bash
git add .dvc/config
git commit -m "Configure GCS as default DVC remote"
```

### Task 3: Version Data & Models Across Iterations

We will version data at path `data/active_data.csv` and the model at path `model/model.joblib`.

Let's create the required directories first.

In [ ]:
import os
os.makedirs("data", exist_ok=True)
os.makedirs("model", exist_ok=True)
print("Data and Model folders prepared.")

#### **Iteration 1: Base Model and Data**

First, we copy the base data `data/raw/iris.csv` to `data/active_data.csv`.

In [3]:
import shutil
import pandas as pd

In [7]:
import shutil
import pandas as pd

# Copy base data
shutil.copy("21f1005023_MLOPS_WEEKLY_ASSIGNMENT/iris_data_raw.csv", "21f1005023_MLOPS_WEEKLY_ASSIGNMENT/data/active_data.csv")
df_v1 = pd.read_csv("21f1005023_MLOPS_WEEKLY_ASSIGNMENT/data/active_data.csv")
print(f"Iteration 1 dataset shape: {df_v1.shape}")

Iteration 1 dataset shape: (150, 5)


Track this base data file `data/active_data.csv` with DVC.

In [ ]:
!dvc add data/active_data.csv

Train Model V1 using the base dataset and save to `model/model.joblib`.

In [10]:
import joblib
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Train Decision Tree model
X = df_v1[["sepal_length", "sepal_width", "petal_length", "petal_width"]]
y = df_v1["species"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model_v1 = DecisionTreeClassifier(max_depth=3, random_state=42)
model_v1.fit(X_train, y_train)
acc = accuracy_score(y_test, model_v1.predict(X_test))
print(f"Model V1 Accuracy: {acc:.4f}")

# Save model file
joblib.dump(model_v1, "21f1005023_MLOPS_WEEKLY_ASSIGNMENT/model/model.joblib")
print("Saved model to model/model.joblib")

Model V1 Accuracy: 1.0000
Saved model to model/model.joblib


Track the model file `model/model.joblib` with DVC.

In [ ]:
!dvc add model/model.joblib

#### **Terminal Action Required**

Run these commands in your terminal to save this version in Git history, tag it, and push data to GCS remote:
```bash
git add data/active_data.csv.dvc model/model.joblib.dvc data/.gitignore model/.gitignore
git commit -m "Version 1.0: Base dataset and model v1"
git tag -a "v1.0" -m "Model and Data Version 1.0"
dvc push
```

In [ ]:
# Alternatively, push tracked data/model from the notebook
!dvc push

#### **Iteration 2: Augmenting Dataset and Training Model V2**

Now we simulate data additions by concatenating the raw dataset, `data/v1/data.csv`, and `data/v2/data.csv` to form a larger dataset.

In [12]:
df_raw = pd.read_csv("21f1005023_MLOPS_WEEKLY_ASSIGNMENT/iris_data_raw.csv")
df_v1_extra = pd.read_csv("21f1005023_MLOPS_WEEKLY_ASSIGNMENT/iris_data_v1.csv")
df_v2_extra = pd.read_csv("21f1005023_MLOPS_WEEKLY_ASSIGNMENT/iris_data_v2.csv")

# Concatenate data
df_augmented = pd.concat([df_raw, df_v1_extra, df_v2_extra], ignore_index=True)
df_augmented.to_csv("21f1005023_MLOPS_WEEKLY_ASSIGNMENT/data/active_data.csv", index=False)

print(f"Iteration 2 dataset shape: {df_augmented.shape}")

Iteration 2 dataset shape: (300, 5)


Track the updated active dataset using DVC.

In [ ]:
!dvc add data/active_data.csv

Retrain Model V2 on the augmented data and save it back to `model/model.joblib`.

In [14]:
X = df_augmented[["sepal_length", "sepal_width", "petal_length", "petal_width"]]
y = df_augmented["species"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model_v2 = DecisionTreeClassifier(max_depth=3, random_state=42)
model_v2.fit(X_train, y_train)
acc = accuracy_score(y_test, model_v2.predict(X_test))
print(f"Model V2 Accuracy: {acc:.4f}")

# Overwrite model
joblib.dump(model_v2, "21f1005023_MLOPS_WEEKLY_ASSIGNMENT/model/model.joblib")
print("Saved updated model to model/model.joblib")

Model V2 Accuracy: 0.9833
Saved updated model to model/model.joblib


Track the updated model file using DVC.

In [ ]:
!dvc add model/model.joblib

#### **Terminal Action Required**

Run these commands in your terminal to save this version in Git history, tag it, and push data to GCS remote:
```bash
git add data/active_data.csv.dvc model/model.joblib.dvc
git commit -m "Version 2.0: Augmented dataset and model v2"
git tag -a "v2.0" -m "Model and Data Version 2.0"
dvc push
```

In [ ]:
# Alternatively, push from the notebook
!dvc push

### Task 4: Switch Between Data/Model Versions

We can now switch between these two versions by checking out different tags in Git and running `dvc checkout`.

#### **1. Switch to Version 1 (v1.0)**

**Terminal commands to run:**
```bash
git checkout v1.0
dvc checkout
```

After running these commands, run the verification cell below to check that the files have been reverted.

In [4]:
# Run this cell after checkout of v1.0
import pandas as pd
import joblib

try:
    df = pd.read_csv("21f1005023_MLOPS_WEEKLY_ASSIGNMENT/data/active_data.csv")
    print(f"Reverted dataset shape: {df.shape} (Expected: 150 rows)")
    
    model = joblib.load("21f1005023_MLOPS_WEEKLY_ASSIGNMENT/model/model.joblib")
    print("Successfully loaded Model V1!")
except Exception as e:
    print(f"Error checking Version 1.0: {e}")

Reverted dataset shape: (150, 5) (Expected: 150 rows)
Successfully loaded Model V1!


#### **2. Switch back to Version 2 (v2.0 - Latest)**

**Terminal commands to run:**
```bash
git checkout week_2
dvc checkout
```

After returning to the `week_2` branch and running DVC checkout, run the verification cell below to check that the files have returned to their latest state.

In [17]:
# Run this cell after returning to week_2
try:
    df = pd.read_csv("21f1005023_MLOPS_WEEKLY_ASSIGNMENT/data/active_data.csv")
    print(f"Restored dataset shape: {df.shape} (Expected: 300 rows)")
    
    model = joblib.load("21f1005023_MLOPS_WEEKLY_ASSIGNMENT/model/model.joblib")
    print("Successfully loaded Model V2 (Latest)!")
except Exception as e:
    print(f"Error checking Version 2.0: {e}")

Restored dataset shape: (300, 5) (Expected: 300 rows)
Successfully loaded Model V2 (Latest)!


### Task 5: Push All Relevant Files

Ensure all `.dvc` files, `.gitignore` files, DVC config, and this notebook are committed and pushed to your remote repository so a reviewer can reproduce your environment.

**Terminal commands to run:**
```bash
git add GA2_notebook.ipynb
git commit -m "Add GA2 notebook with walkthrough"
git push origin week_2 --tags
```